# 04 — Batch Screening

Full batch analysis of all samples in the experiment:

1. Run DK batch fitting (fast screening)
2. Quality flags & filtering
3. KD ranking & distributions
4. Sensorgram gallery of top hits
5. Export results to CSV for downstream use (Boltz2 / OpenFold)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sensefit import load_cxw, batch_fit, flag_poor_fits
from sensefit.models import (
    double_reference, build_weight_mask, is_nonspecific_binder,
    simulate_sensorgram
)
from sensefit.direct_kinetics import fit_sample as dk_fit_sample

CXW = '../20250826_DENV-2 NS2B3 Binding Assay.cxw'

## 1. DK Batch Fit

In [ ]:
df, data = batch_fit(CXW, mode='dk')
df = flag_poor_fits(df)

samples = data['samples']
dmso_cals = data['dmso_cals']
blanks = data['blanks']

n_total = len(df)
n_nsb = df['nonspecific'].sum()
n_flagged = df['flag'].sum()
n_valid = n_total - n_flagged

print(f'Total samples:          {n_total}')
print(f'Non-specific binders:   {n_nsb}')
print(f'Flagged (total):        {n_flagged}')
print(f'Valid fits:             {n_valid}')

## 2. Flag Breakdown

In [ ]:
# Count each flag reason
all_reasons = []
for reasons_str in df['flag_reason']:
    if reasons_str:
        all_reasons.extend(reasons_str.split('; '))

if all_reasons:
    reason_counts = pd.Series(all_reasons).value_counts()
    print('Flag breakdown:')
    for reason, count in reason_counts.items():
        print(f'  {reason:25s}  {count}')
else:
    print('No flags raised.')

## 3. KD Distribution — Valid Fits

In [ ]:
valid = df[~df['flag']].copy()

if len(valid) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    ax = axes[0]
    ax.hist(valid['KD_uM'], bins=20, color='steelblue', edgecolor='white')
    ax.set_xlabel('KD (µM)')
    ax.set_ylabel('Count')
    ax.set_title(f'KD Distribution (n={len(valid)})')
    ax.grid(True, alpha=0.3)

    # Log-scale histogram
    ax = axes[1]
    log_kd = np.log10(valid['KD_uM'].clip(lower=1e-3))
    ax.hist(log_kd, bins=20, color='indianred', edgecolor='white')
    ax.set_xlabel('log₁₀(KD / µM)')
    ax.set_ylabel('Count')
    ax.set_title('KD Distribution (log scale)')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'KD median: {valid["KD_uM"].median():.2f} µM')
    print(f'KD range:  {valid["KD_uM"].min():.3f} – {valid["KD_uM"].max():.1f} µM')
else:
    print('No valid fits to plot.')

## 4. KD Ranking — All Valid Compounds

In [ ]:
if len(valid) > 0:
    ranked = valid.sort_values('KD_uM')
    cols = ['compound', 'concentration_uM', 'ka', 'kd', 'KD_uM', 'Rmax',
            'sigma_res']
    print(ranked[cols].to_string(index=False))

In [ ]:
if len(valid) > 0:
    ranked = valid.sort_values('KD_uM')

    fig, ax = plt.subplots(figsize=(10, max(4, len(ranked) * 0.3)))
    y = range(len(ranked))
    ax.barh(y, ranked['KD_uM'].values, color='steelblue', height=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['compound']} ({r['concentration_uM']:.0f}µM)"
                        for _, r in ranked.iterrows()], fontsize=7)
    ax.set_xlabel('KD (µM)')
    ax.set_title('Compounds ranked by KD (lower = tighter binder)')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. Sensorgram Gallery — Top Hits

Plot the double-referenced sensorgram + DK model overlay for the top 6 binders.

In [ ]:
if len(valid) > 0:
    top_n = min(6, len(valid))
    top = valid.sort_values('KD_uM').head(top_n)

    # Map cycle_index → sample dict
    sample_lookup = {s['index']: s for s in samples}

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()

    for i, (_, row) in enumerate(top.iterrows()):
        if i >= len(axes):
            break
        ax = axes[i]
        s = sample_lookup[row['cycle_index']]

        # Refit to get arrays
        try:
            result = dk_fit_sample(s, dmso_cals, blanks=blanks)
            t = result['t']
            sig = result['signal']
            R_sim = simulate_sensorgram(t, result['ka'], result['kd'],
                                        result['Rmax'], result['c_func'])

            ax.plot(t, sig, 'k-', lw=0.4, alpha=0.5)
            ax.plot(t, R_sim, 'r-', lw=1)
        except Exception:
            ax.text(0.5, 0.5, 'Fit failed', transform=ax.transAxes,
                    ha='center', va='center')

        ax.set_title(f'{row["compound"]}\nKD={row["KD_uM"]:.2f} µM  '
                     f'({row["concentration_uM"]:.0f} µM)', fontsize=9)
        ax.grid(True, alpha=0.3)
        if i >= 3:
            ax.set_xlabel('Time (s)')
        if i % 3 == 0:
            ax.set_ylabel('pg/mm²')

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Top binders — sensorgram + Langmuir fit', fontsize=13)
    plt.tight_layout()
    plt.show()

## 6. Full Results Table

In [ ]:
cols_display = ['compound', 'concentration_uM', 'ka', 'kd', 'KD_uM',
                'Rmax', 'sigma_res', 'nonspecific', 'success',
                'flag', 'flag_reason']
display_cols = [c for c in cols_display if c in df.columns]
df[display_cols]

## 7. Export to CSV

Save the full results and a curated valid-only table for downstream ML pipelines.

In [ ]:
# Full results
df.to_csv('batch_results_all.csv', index=False)
print(f'Saved batch_results_all.csv  ({len(df)} rows)')

# Valid fits only — for KD training data
if len(valid) > 0:
    export_cols = ['compound', 'concentration_uM', 'mw', 'ka', 'kd',
                   'KD_uM', 'Rmax', 'sigma_res']
    export_cols = [c for c in export_cols if c in valid.columns]
    valid[export_cols].to_csv('batch_results_valid.csv', index=False)
    print(f'Saved batch_results_valid.csv  ({len(valid)} rows)')

# NSB list
nsb_df = df[df['nonspecific'] == True][['compound', 'concentration_uM', 'ref_dissoc']]
if len(nsb_df) > 0:
    nsb_df.to_csv('nonspecific_binders.csv', index=False)
    print(f'Saved nonspecific_binders.csv  ({len(nsb_df)} rows)')

## 8. Summary Statistics

In [ ]:
print('=' * 50)
print('BATCH SCREENING SUMMARY')
print('=' * 50)
print(f'Experiment:             {CXW}')
print(f'Total samples:          {n_total}')
print(f'Non-specific binders:   {n_nsb} ({n_nsb/n_total*100:.0f}%)')
print(f'Valid fits:             {n_valid} ({n_valid/n_total*100:.0f}%)')
print(f'Flagged (other):        {n_flagged - n_nsb}')
if len(valid) > 0:
    print(f'\nKD statistics (valid fits):')
    print(f'  Median:  {valid["KD_uM"].median():.2f} µM')
    print(f'  Mean:    {valid["KD_uM"].mean():.2f} µM')
    print(f'  Min:     {valid["KD_uM"].min():.3f} µM')
    print(f'  Max:     {valid["KD_uM"].max():.1f} µM')
    best = valid.loc[valid['KD_uM'].idxmin()]
    print(f'\nBest binder: {best["compound"]} — KD = {best["KD_uM"]:.3f} µM')